# BackstageCommercials Pipeline Notebook

Complete end-to-end pipeline for embedding products into video with photorealistic blending and Amazon integration.

**Requirements:**
- A100 GPU (40GB VRAM) for full-precision FLUX
- `NOVA_API_KEY` and `NOVA_ACT_API_KEY` in `.env`
- Input: video file + product image + product description

**Output:**
- Video with embedded product + original audio
- `video_config.txt` ready for frontend
- Product image asset

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image

load_dotenv()

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

import select_frame
import insert_product
import flux
import generate_video
from agent_find_product import ask_nova_search_product_agent

print('Imports complete')

## 2. User Inputs

Provide the video, product image, and a short description of the product.

In [ ]:
# === USER INPUTS ===
VIDEO_PATH = "./input_video.mp4"          # Path to your video file
PRODUCT_IMAGE_PATH = "./product.png"       # Path to product image (PNG with transparent bg recommended)
PRODUCT_DESCRIPTION = "coffee jar"         # Short description for placement and Amazon search
SHOW_ID = "myshow"                         # Unique ID matching baseShows in frontend

print(f"Video: {VIDEO_PATH}")
print(f"Product: {PRODUCT_IMAGE_PATH}")
print(f"Description: {PRODUCT_DESCRIPTION}")
print(f"Show ID: {SHOW_ID}")

## 3. Step 1: Select Best Frame for Placement

Detects stable shots and asks Nova to pick the best frame for product insertion.

In [ ]:
print("=== SELECTING BEST SHOT ===")
shot_result = select_frame.find_best_product_placement_shot(
    video_path=VIDEO_PATH
)

print(json.dumps(shot_result, indent=2))

begin_frame = shot_result["best_shot_start_frame"]
end_frame = shot_result["best_shot_end_frame"]
fps = shot_result.get("fps", 30.0)

print(f"\nBest shot: frames {begin_frame} - {end_frame}")

## 4. Extract First Frame of Selected Shot

In [ ]:
import cv2

video = cv2.VideoCapture(VIDEO_PATH)
if not video.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

video.set(cv2.CAP_PROP_POS_FRAMES, begin_frame)
success, frame = video.read()
if not success:
    raise RuntimeError(f"Failed to read frame {begin_frame}")

first_frame_path = "first_frame.png"
cv2.imwrite(first_frame_path, frame)
video.release()

print(f"Saved first frame: {first_frame_path}")

## 5. Step 2: Intelligent Product Placement (Nova)

Nova placement model proposes bbox → paste → evaluation model validates → repeat until valid or max iters.

In [ ]:
print("=== RECURSIVE PLACEMENT ===")
final_placement_image, bbox_coords = insert_product.recursive_placement(
    background=first_frame_path,
    product=PRODUCT_IMAGE_PATH,
    product_description=PRODUCT_DESCRIPTION,
    max_iters=6
)

print(f"Final placement image: {final_placement_image}")
print(f"Bbox coords (normalized): {bbox_coords}")

# Convert normalized bbox to pixel coordinates for generate_video
bg = Image.open(first_frame_path)
W, H = bg.size
pixel_bbox = (
    int(bbox_coords["x"] * W),
    int(bbox_coords["y"] * H),
    int((bbox_coords["x"] + bbox_coords["width"]) * W),
    int((bbox_coords["y"] + bbox_coords["height"]) * H)
)
print(f"Bbox coords (pixels): {pixel_bbox}")

## 6. Step 3: Photorealistic Blending (Full-Precision FLUX + LoRA)

**Full precision mode** — no quantization. Requires ~24GB VRAM (A100 40GB is sufficient).

In [ ]:
print("=== FLUX PRODUCT PLACEMENT (FULL PRECISION) ===")
flux_output, flux_filename = flux.place_product(
    scene_image=first_frame_path,
    reference=PRODUCT_IMAGE_PATH,
    bbox=bbox_coords,
    product_description=PRODUCT_DESCRIPTION
)

print(f"FLUX output saved: {flux_filename}")
print(f"FLUX image: flux_output.png")

## 7. Step 4: Generate Final Video with Segmentation

In [ ]:
print("=== GENERATING VIDEO ===")
output_video_path = "product_placement_output.mp4"

generate_video.generate_video(
    filename="flux_output.png",
    video_path=VIDEO_PATH,
    begin_frame=begin_frame,
    end_frame=end_frame,
    product_bbox=pixel_bbox,
    output_path=output_video_path
)

print(f"Generated video: {output_video_path}")

## 8. Step 5: Restore Original Audio (ffmpeg stream copy)

Lossless audio mux — no re-encoding, frame indices unchanged.

In [ ]:
print("=== RESTORING AUDIO ===")
final_video_with_audio = "final_output_with_audio.mp4"

cmd = [
    "ffmpeg", "-y",
    "-i", output_video_path,
    "-i", VIDEO_PATH,
    "-c", "copy",
    "-map", "0:v:0",
    "-map", "1:a:0",
    final_video_with_audio
]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("FFMPEG stderr:", result.stderr)
    raise RuntimeError("Audio restoration failed")

print(f"Final video with audio: {final_video_with_audio}")

## 9. Step 6: Fetch Amazon Product Metadata (Nova Agent)

In [ ]:
print("=== FETCHING AMAZON METADATA ===")

nova_client = OpenAI(
    api_key=os.getenv("NOVA_API_KEY"),
    base_url="https://api.nova.amazon.com/v1"
)

# Encode first frame as data URL for Nova
import base64
import mimetypes

def to_data_url(path, mime_type="image/jpeg"):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"

frame_data_url = to_data_url(first_frame_path)
text_prompt = f"Find this {PRODUCT_DESCRIPTION} on Amazon"

amazon_result = ask_nova_search_product_agent(
    client=nova_client,
    input_image=frame_data_url,
    text_prompt=text_prompt
)

print("Amazon search result:")
print(json.dumps(amazon_result, indent=2))

amazon_link = amazon_result.get("amazon_link", ["https://amazon.com"])[0] if amazon_result.get("amazon_link") else "https://amazon.com"
item_description = amazon_result.get("item_description", PRODUCT_DESCRIPTION)

## 10. Step 7: Generate Frontend Config Line

Format matches `parseVideoConfig()` in `App.jsx`.

In [ ]:
print("=== GENERATING CONFIG ===")

# For demo purposes, use placeholder values for price/rating
# In production, parse these from amazon_result or scrape the Amazon page
PRODUCT_TITLE = item_description[:50]
PRODUCT_PRICE = "$19.99"
PRODUCT_RATING = "4"
PRODUCT_URL = amazon_link

config_line = (
    f"{SHOW_ID}|"
    f"/videos/{final_video_with_audio}|"
    f"{begin_frame}|"
    f"{end_frame}|"
    f"/products/{Path(PRODUCT_IMAGE_PATH).name}|"
    f"{PRODUCT_TITLE}|"
    f"{item_description}|"
    f"{PRODUCT_PRICE}|"
    f"{PRODUCT_RATING}|"
    f"{PRODUCT_URL}"
)

print("\nAdd this line to frontend/public/video_config.txt:")
print(config_line)

# Also write to a standalone file
with open("video_config_line.txt", "w") as f:
    f.write(config_line + "\n")
print("\nSaved to: video_config_line.txt")

## 11. Asset Staging Instructions

Copy these files to the frontend:

In [ ]:
print("=== STAGING INSTRUCTIONS ===")
print(f"\n1. Copy video: {final_video_with_audio}")
print(f"   → frontend/public/videos/{final_video_with_audio}")
print(f"\n2. Copy product image: {PRODUCT_IMAGE_PATH}")
print(f"   → frontend/public/products/{Path(PRODUCT_IMAGE_PATH).name}")
print(f"\n3. Append to frontend/public/video_config.txt:")
print(f"   {config_line}")
print("\n4. Ensure frontend is running: cd frontend/prime-video-ui && npm run dev")

---

**Pipeline complete.** The generated video contains the photorealistically embedded product, original audio is preserved, and the config line is ready for the frontend to display timed purchase popups.